# Llama-3.1-8B FP16 vs GPTQ-INT4 mini-pilot (Kaggle T4 ×2)

Same pipeline as `mini_pilot/run_1b_fp16_vs_int4.py`: ConflictQA items, three generations per item (closed-book / agrees / conflicts), **shared double-filter MR★**, 1000-draw bootstrap CI, mixed-logit AME. Only the model is 8B.

## Before you hit Run All

1. **Accelerator:** GPU T4 ×2 (not P100, not single T4).
2. **Internet:** on.
3. **Hugging Face license:** accept [Llama-3.1-8B-Instruct](https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct) while logged in.
4. **Kaggle secret:** Add-ons → Secrets → `HF_TOKEN` (write access not required; gated-read is).
5. **Attach this repo as a Dataset** (exclude `.venv/`). Name does not matter — cell 4 searches `/kaggle/input` for `common/models.py`.
6. Optional: attach a previous `checkpoint_8b.jsonl` dataset to resume after a timeout.

## VRAM (best estimate)

| Precision | Weights | Fits **one** T4 (16 GB)? | Fits **two** T4s via `device_map="auto"`? |
|---|---|---|---|
| **FP16** `Llama-3.1-8B-Instruct` | ~16.1 GiB (+ CUDA/activations) | **No.** Weights alone exceed usable 16 GB after the CUDA context. | **Yes.** Typical split ~8–9 GiB/GPU + ~1–2 GiB activations. This notebook uses `device_map="auto"` and forbids CPU/disk offload. |
| **GPTQ INT4** (`GPTQ_8B_MODEL_ID` in `common/models.py`) | ~5.5–7 GiB runtime | **Yes, comfortably.** | Unnecessary. We **pin `{"": 0}`**. `device_map="auto"` *is* supported for GPTQ (HF shards at **layer** boundaries; QuantLinear is not split), but after FP16 used both GPUs it is safer to keep INT4 on GPU 0. |

T4 has **no useful native bfloat16** — loads stay `float16`.

Wall-clock for n=250 (3 gens × 32 new tokens): roughly **3–6 hours** plus ~15 min of downloads. Fits the 12 h session if you start promptly. Checkpoint every 20 items to `/kaggle/working`.

## Cell 1 — packages

Do **not** reinstall `torch`. Kaggle already has a CUDA build. Restart the kernel if the next cell cannot import `auto_gptq`.

In [ ]:
# Kaggle already ships torch + CUDA. Only add what the mini-pilot needs.
import os, sys, subprocess

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HOME"] = "/tmp/hf_home"
os.environ["TRANSFORMERS_CACHE"] = "/tmp/hf_home"
os.environ["HF_HUB_DISABLE_XET"] = "1"

pkgs = [
    "transformers==4.48.0",
    "accelerate==1.0.1",
    "optimum==1.23.3",
    "peft==0.10.0",          # required by auto-gptq 0.7.x
    "datasets==3.1.0",
    "huggingface-hub==0.26.2",
    "safetensors==0.4.5",
    "statsmodels==0.14.4",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

# auto-gptq: try the CUDA wheel index first, then a plain pip fallback.
rc = subprocess.call(
    [
        sys.executable, "-m", "pip", "install", "-q", "auto-gptq==0.7.1",
        "--extra-index-url", "https://huggingface.github.io/autogptq-index/whl/cu121/",
    ]
)
if rc != 0:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "auto-gptq==0.7.1"])

import torch, transformers, auto_gptq, peft
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), "ngpu", torch.cuda.device_count())
print("transformers", transformers.__version__, "peft", peft.__version__, "auto_gptq", auto_gptq.__version__)

## Cell 2 — Hugging Face login via Kaggle secrets (never paste a token into a cell)

In [ ]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=token, add_to_git_credential=False)
print("HF login ok")

## Cell 3 — GPU / disk sanity check

In [ ]:
import shutil, torch

assert torch.cuda.is_available(), "Enable GPU T4 x2 in the notebook settings."
n = torch.cuda.device_count()
print(f"CUDA devices: {n}")
for i in range(n):
    p = torch.cuda.get_device_properties(i)
    print(f"  [{i}] {p.name} {p.total_memory/1024**3:.1f} GiB")
if n < 2:
    print("WARNING: FP16 8B will not fit on a single 16GB T4. Switch accelerator to GPU T4 x2.")

for label, path in [("working", "/kaggle/working"), ("tmp", "/tmp"), ("hf_home", "/tmp/hf_home")]:
    total, used, free = shutil.disk_usage(path if path != "/tmp/hf_home" else "/tmp")
    print(f"disk {label}: free {free/1024**3:.1f} GiB")

## Cell 4 — put this repo on `sys.path`

Attach the project (folder that contains `common/` and `mini_pilot/`) as a Kaggle Dataset. This cell copies it into `/kaggle/working/tt` so outputs stay writable.

If you did not attach a dataset, set `GITHUB_URL` to a cloneable URL instead.

In [ ]:
from pathlib import Path
import shutil, sys, subprocess, os

GITHUB_URL = ""  # e.g. "https://github.com/YOU/tt.git"  — leave blank if you attached a Dataset

def find_repo() -> Path | None:
    hits = list(Path("/kaggle/input").rglob("common/models.py")) if Path("/kaggle/input").exists() else []
    hits += list(Path("/kaggle/working").glob("tt/common/models.py"))
    return hits[0].parents[1] if hits else None

REPO = Path("/kaggle/working/tt")
src = find_repo()
if src is None and GITHUB_URL:
    subprocess.check_call(["git", "clone", "--depth", "1", GITHUB_URL, str(REPO)])
    src = REPO
if src is None:
    raise SystemExit(
        "Could not find common/models.py. Attach this repo as a Kaggle Dataset "
        "(zip without .venv) or set GITHUB_URL."
    )
if src.resolve() != REPO.resolve():
    if REPO.exists():
        shutil.rmtree(REPO)
    shutil.copytree(src, REPO, ignore=shutil.ignore_patterns(".venv", "__pycache__", ".git"))
    print(f"Copied repo {src} -> {REPO}")
else:
    print(f"Using repo at {REPO}")

sys.path.insert(0, str(REPO))
os.chdir(REPO)
print("cwd", Path.cwd())
print("has runner", (REPO / "mini_pilot" / "run_8b_fp16_vs_int4.py").exists())

## Cell 5 — run config (`n_items` and analysis flags match the 1B runner)

In [ ]:
from pathlib import Path

# Keep this equal to the 1B mini-pilot if you want a head-to-head (last 1B run used 250).
N_ITEMS = 250
CHECKPOINT_EVERY = 20
MAX_NEW_TOKENS = 32
N_BOOT = 1000
SEED = 42
QUANT_BACKEND = "gptq"  # do not switch to bnb_nf4 for the paper table

OUT_DIR = Path("/kaggle/working/mini_pilot/outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_PATH = Path("/kaggle/working/conflictqa_eval.jsonl")
print(dict(N_ITEMS=N_ITEMS, CHECKPOINT_EVERY=CHECKPOINT_EVERY, OUT_DIR=str(OUT_DIR)))

## Cell 6 — ConflictQA JSONL

Prefers the same `conflictqa_pilot250_popular.jsonl` that shipped with the repo (identical item set as the 1B pilot). Falls back to exporting from Hugging Face with the same popularity filter.

In [ ]:
from pathlib import Path
import shutil

candidates = list(Path("/kaggle/working/tt").rglob("conflictqa_pilot250_popular.jsonl"))
candidates += list(Path("/kaggle/input").rglob("conflictqa_pilot250_popular.jsonl")) if Path("/kaggle/input").exists() else []
if candidates:
    shutil.copy(candidates[0], DATA_PATH)
    print(f"Copied {candidates[0]} -> {DATA_PATH}")
else:
    from common.conflictqa import export_conflictqa_from_hf
    n = export_conflictqa_from_hf(
        DATA_PATH,
        max_items=N_ITEMS,
        top_by_popularity=True,
        dedupe_questions=True,
    )
    print(f"Exported {n} rows from osunlp/ConflictQA")

n_lines = sum(1 for _ in DATA_PATH.open())
print(f"{DATA_PATH} has {n_lines} rows")

## Cell 7 — run FP16 then GPTQ-INT4 (resumable)

Re-run this cell after a disconnect. It skips item ids already in `checkpoint_8b.jsonl`. If the previous session died, **Save Version** or download that JSONL and attach it as a Dataset before the next session.

FP16 uses `device_map="auto"` (both T4s). GPTQ is pinned to GPU 0.

In [ ]:
from mini_pilot.run_8b_fp16_vs_int4 import default_args, run_pilot

args = default_args(
    data=DATA_PATH,
    n_items=N_ITEMS,
    max_new_tokens=MAX_NEW_TOKENS,
    seed=SEED,
    quant_backend=QUANT_BACKEND,
    out_dir=OUT_DIR,
    n_boot=N_BOOT,
    checkpoint_every=CHECKPOINT_EVERY,
    checkpoint_path=OUT_DIR / "checkpoint_8b.jsonl",
    resume_from=Path("/kaggle/input"),
    fp16_device_map="auto",
    int4_device_map="gpu0",
)
print(args)
run_pilot(args)

## Cell 8 — after the run

Download from `/kaggle/working/mini_pilot/outputs/`:

- `checkpoint_8b.jsonl` — generations (also the resume file)
- `summary_*.csv`, `stats_*.json`, `run_meta_*.json`, `detail_*.jsonl`, `mr_vs_mrstar_*.png`

**Across sessions:** Kaggle `/kaggle/working` is wiped when the session ends unless you Save Version. Mid-run, periodically download `checkpoint_8b.jsonl`. Next session: upload it as a Dataset; cell 7 will pick it up from `/kaggle/input`.

If the kernel died mid-cell, just re-run cell 7. Completed `(id, precision)` rows are skipped.

In [ ]:
from pathlib import Path
out = Path("/kaggle/working/mini_pilot/outputs")
for p in sorted(out.glob("*")):
    print(f"{p.name:40s} {p.stat().st_size/1024:8.1f} KiB")